# Wikidata Bootstrap: Simple Entity Linking

This notebook bootstraps a tiny local knowledge base from Wikidata and then uses exact string matching to annotate text with Wikidata URIs.

Default domain: living people currently holding a political office in Ireland.

## Important idea

The local KB is not hidden. It is just:

1. a SPARQL query,
2. the rows returned by Wikidata,
3. a little Python that groups those rows,
4. and a surface-form index built from the labels.

If you want a different KB, edit the `.sparql` query file or point `query_path` at another `.sparql` file, then rerun the notebook.

In [12]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / 'src'))

from wikidata_lab.agentic_linking import AgenticLinkerTemplate
from wikidata_lab.candidate_retrieval import SimpleVectorStoreTemplate
from wikidata_lab.wikidata import (
    DEFAULT_BOOTSTRAP_QUERY_PATH,
    annotate_text,
    build_surface_form_index,
    fetch_knowledge_base_from_query,
    load_sparql_query,
)


In [13]:
query_path = DEFAULT_BOOTSTRAP_QUERY_PATH
print(query_path)
query = load_sparql_query(query_path)
print(query)

C:\Users\daksh\Documents\agentic-entity-linking-lab\queries\current_living_irish_office_holders.sparql
SELECT DISTINCT ?person ?personLabel WHERE {
  ?person wdt:P31 wd:Q5;
          wdt:P27 wd:Q27;
          wdt:P106 wd:Q639669.
  FILTER NOT EXISTS { ?person wdt:P570 ?dateOfDeath }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?personLabel


## Change the KB here

Best practice: keep your SPARQL in a clearly named `.sparql` file.

For this notebook, edit the file printed above if you want to:

1. change country,
2. change occupation,
3. narrow to a different office,
4. or ask for a different domain entirely.

After editing the file, rerun the query-loading cell and then the next cell.

In [14]:
raw_rows, kb = fetch_knowledge_base_from_query(query)
print(f'raw rows: {len(raw_rows)}')
print(f'entities in local KB: {len(kb)}')

raw rows: 246
entities in local KB: 246


In [15]:
raw_rows[:5]

[{'person': 'http://www.wikidata.org/entity/Q130314253',
  'personLabel': 'Q130314253'},
 {'person': 'http://www.wikidata.org/entity/Q133399071',
  'personLabel': 'Q133399071'},
 {'person': 'http://www.wikidata.org/entity/Q134304625',
  'personLabel': 'Q134304625'},
 {'person': 'http://www.wikidata.org/entity/Q2362745',
  'personLabel': 'Aidan Coffey'},
 {'person': 'http://www.wikidata.org/entity/Q4697412',
  'personLabel': 'Aindrias Stack'}]

In [16]:
kb[:5]

[{'uri': 'http://www.wikidata.org/entity/Q2362745',
  'label': 'Aidan Coffey',
  'surface_forms': ['Aidan Coffey'],
  'current_offices': [],
  'metadata': {'wikidata_id': 'Q2362745'}},
 {'uri': 'http://www.wikidata.org/entity/Q4697412',
  'label': 'Aindrias Stack',
  'surface_forms': ['Aindrias Stack'],
  'current_offices': [],
  'metadata': {'wikidata_id': 'Q4697412'}},
 {'uri': 'http://www.wikidata.org/entity/Q91874932',
  'label': 'Allie Sherlock',
  'surface_forms': ['Allie Sherlock'],
  'current_offices': [],
  'metadata': {'wikidata_id': 'Q91874932'}},
 {'uri': 'http://www.wikidata.org/entity/Q138710410',
  'label': 'Amber Excellent',
  'surface_forms': ['Amber Excellent'],
  'current_offices': [],
  'metadata': {'wikidata_id': 'Q138710410'}},
 {'uri': 'http://www.wikidata.org/entity/Q71952698',
  'label': 'Amer Nejma',
  'surface_forms': ['Amer Nejma'],
  'current_offices': [],
  'metadata': {'wikidata_id': 'Q71952698'}}]

## Build a surface-form index

By default, each entity uses its Wikidata label as the surface form.

That keeps the baseline explicit and easy to understand.

If you want more surface forms, add aliases manually or extend the query.

In [17]:
surface_form_index = build_surface_form_index(kb)
list(surface_form_index.items())[:5]

[('aidan coffey',
  [{'uri': 'http://www.wikidata.org/entity/Q2362745',
    'label': 'Aidan Coffey',
    'surface_forms': ['Aidan Coffey'],
    'current_offices': [],
    'metadata': {'wikidata_id': 'Q2362745'}}]),
 ('aindrias stack',
  [{'uri': 'http://www.wikidata.org/entity/Q4697412',
    'label': 'Aindrias Stack',
    'surface_forms': ['Aindrias Stack'],
    'current_offices': [],
    'metadata': {'wikidata_id': 'Q4697412'}}]),
 ('allie sherlock',
  [{'uri': 'http://www.wikidata.org/entity/Q91874932',
    'label': 'Allie Sherlock',
    'surface_forms': ['Allie Sherlock'],
    'current_offices': [],
    'metadata': {'wikidata_id': 'Q91874932'}}]),
 ('amber excellent',
  [{'uri': 'http://www.wikidata.org/entity/Q138710410',
    'label': 'Amber Excellent',
    'surface_forms': ['Amber Excellent'],
    'current_offices': [],
    'metadata': {'wikidata_id': 'Q138710410'}}]),
 ('amer nejma',
  [{'uri': 'http://www.wikidata.org/entity/Q71952698',
    'label': 'Amer Nejma',
    'surface_fo

In [18]:
sample_people = [entity['label'] for entity in kb[:3]]
sample_text = (
    f"{sample_people[0]} met {sample_people[1]} in Dublin while {sample_people[2]} "
    "spoke about the coalition."
)
annotations = annotate_text(sample_text, surface_form_index)
annotations

[{'start': 0,
  'end': 12,
  'text': 'Aidan Coffey',
  'normalized_surface_form': 'aidan coffey',
  'uri': 'http://www.wikidata.org/entity/Q2362745',
  'candidate_uris': ['http://www.wikidata.org/entity/Q2362745'],
  'candidate_labels': ['Aidan Coffey']},
 {'start': 17,
  'end': 31,
  'text': 'Aindrias Stack',
  'normalized_surface_form': 'aindrias stack',
  'uri': 'http://www.wikidata.org/entity/Q4697412',
  'candidate_uris': ['http://www.wikidata.org/entity/Q4697412'],
  'candidate_labels': ['Aindrias Stack']},
 {'start': 48,
  'end': 62,
  'text': 'Allie Sherlock',
  'normalized_surface_form': 'allie sherlock',
  'uri': 'http://www.wikidata.org/entity/Q91874932',
  'candidate_uris': ['http://www.wikidata.org/entity/Q91874932'],
  'candidate_labels': ['Allie Sherlock']}]

## Annotate the example file from the repository

This is the same baseline pipeline, but using a real example document from `data/` and passing its contents into `annotate_text(...)` as a function argument.

In [19]:
example_path = REPO_ROOT / 'data' / 'example_current_irish_office_holders.txt'
example_text = example_path.read_text(encoding='utf-8')
print(example_text)

annotate_text(example_text, surface_form_index)

Aengus Ó Snodaigh met Aisling Dempsey in Dublin after Alan Dillon briefed Alice Mary Higgins and Aidan Farrelly on the proposed legislation.



[]

## Alias expansion exercise

Try adding extra surface forms to the local KB. This is one of the easiest first extensions.

Examples:

1. ASCII variants without accents
2. common shortened names
3. party-title variants
4. aliases fetched from Wikidata in a second query

In [20]:
mary_lou_entity = next((entity for entity in kb if entity['label'] == 'Mary Lou McDonald'), None)
manual_aliases = {}
if mary_lou_entity is not None:
    manual_aliases[mary_lou_entity['uri']] = ['Mary Lou']

surface_form_index_with_aliases = build_surface_form_index(
    kb,
    extra_surface_forms_by_uri=manual_aliases,
)

annotate_text('Mary Lou addressed supporters.', surface_form_index_with_aliases)

[]

## Vector retrieval template

The repository includes a dependency-free scaffold for vector retrieval.

To use it, plug in an embedding provider with `embed_texts(texts)`.

In [21]:
class DemoEmbeddingProvider:
    def embed_texts(self, texts):
        vectors = []
        for text in texts:
            lowered = text.casefold()
            vectors.append([
                float('mary' in lowered),
                float('martin' in lowered),
                float(len(lowered)),
            ])
        return vectors

vector_store = SimpleVectorStoreTemplate(DemoEmbeddingProvider())
vector_store.index_entities(kb)
demo_mention = kb[0]['label']
vector_store.retrieve(demo_mention, top_k=5)

[EntityCandidate(uri='http://www.wikidata.org/entity/Q2362745', label='Aidan Coffey', score=1.0, metadata={'current_offices': [], 'wikidata_id': 'Q2362745'}),
 EntityCandidate(uri='http://www.wikidata.org/entity/Q4697412', label='Aindrias Stack', score=1.0, metadata={'current_offices': [], 'wikidata_id': 'Q4697412'}),
 EntityCandidate(uri='http://www.wikidata.org/entity/Q91874932', label='Allie Sherlock', score=1.0, metadata={'current_offices': [], 'wikidata_id': 'Q91874932'}),
 EntityCandidate(uri='http://www.wikidata.org/entity/Q138710410', label='Amber Excellent', score=1.0, metadata={'current_offices': [], 'wikidata_id': 'Q138710410'}),
 EntityCandidate(uri='http://www.wikidata.org/entity/Q71952698', label='Amer Nejma', score=1.0, metadata={'current_offices': [], 'wikidata_id': 'Q71952698'})]

In [24]:
import json
from wikidata_lab.sentence_transformer_embeddings import SentenceTransformerEmbeddingProvider

# Replace the fake demo embedder with a real sentence transformer
provider = SentenceTransformerEmbeddingProvider()
st_vector_store = SimpleVectorStoreTemplate(provider)
st_vector_store.index_entities(kb)

# Test 1: full name
mention_full = "Allie Sherlock"
candidates_full = st_vector_store.retrieve(mention_full, top_k=5)
print(f"Results for full name: '{mention_full}'")
print(json.dumps([c.__dict__ for c in candidates_full], indent=2))

print("\n" + "="*50 + "\n")

# Test 2: nickname only
mention_short = "Allie"
candidates_short = st_vector_store.retrieve(mention_short, top_k=5)
print(f"Results for nickname: '{mention_short}'")
print(json.dumps([c.__dict__ for c in candidates_short], indent=2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Results for full name: 'Allie Sherlock'
[
  {
    "uri": "http://www.wikidata.org/entity/Q91874932",
    "label": "Allie Sherlock",
    "score": 0.999999999999873,
    "metadata": {
      "current_offices": [],
      "wikidata_id": "Q91874932"
    }
  },
  {
    "uri": "http://www.wikidata.org/entity/Q284227",
    "label": "Lisa Hannigan",
    "score": 0.485729488303449,
    "metadata": {
      "current_offices": [],
      "wikidata_id": "Q284227"
    }
  },
  {
    "uri": "http://www.wikidata.org/entity/Q7440774",
    "label": "Seamus Moore",
    "score": 0.485723099309043,
    "metadata": {
      "current_offices": [],
      "wikidata_id": "Q7440774"
    }
  },
  {
    "uri": "http://www.wikidata.org/entity/Q138710488",
    "label": "Lenny Hennessy",
    "score": 0.4671026158044284,
    "metadata": {
      "current_offices": [],
      "wikidata_id": "Q138710488"
    }
  },
  {
    "uri": "http://www.wikidata.org/entity/Q6396166",
    "label": "Kevin Doherty",
    "score": 0.445651595

## Context-driven agentic linker template

This follows the same structure as the reference SNOMED project:

1. retrieve candidates,
2. extract local context,
3. build a prompt,
4. call an injected LLM backend.

In [25]:
agent = AgenticLinkerTemplate()
candidate_list = vector_store.retrieve(demo_mention, top_k=3)
prompt = agent.build_prompt(
    full_text=f'{demo_mention} spoke in the Dail this afternoon.',
    mention_text=demo_mention,
    start=0,
    end=len(demo_mention),
    candidates=candidate_list,
)
print(prompt)

Task: link one mention to the best Wikidata entity.
Mention: Aidan Coffey
Span: 0-12
Context: ...[Aidan Coffey] spoke in the Dail this afternoon....

Candidates:
1. Aidan Coffey | http://www.wikidata.org/entity/Q2362745 | score=1.0000 | offices=
2. Aindrias Stack | http://www.wikidata.org/entity/Q4697412 | score=1.0000 | offices=
3. Allie Sherlock | http://www.wikidata.org/entity/Q91874932 | score=1.0000 | offices=

Return JSON only.


## Suggested student exercises

1. Change the SPARQL query to a new domain.
2. Add alias expansion.
3. Improve the spotter to handle punctuation and accents.
4. Add a better candidate retrieval stage.
5. Plug in an LLM backend and build a context-aware linker.
6. Create a tiny evaluation set and score your linker.